In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, LSTM, RepeatVector, TimeDistributed, Dense, Masking
from tensorflow.keras.callbacks import EarlyStopping

# Secure reproducible seeds
np.random.seed(42)
tf.random.set_seed(42)

def get_system_ram_gb():
    return psutil.virtual_memory().available / (1024 ** 3)

print(f"[SETUP] Initializing Engine.")

[SETUP] Initializing Engine.


## Data Loading and Cleaning

In [9]:
TARGET_ROWS = 250000  # Balanced safe size
CSV_FILE = 'friday.csv'

print(f"[DATA] Loading {TARGET_ROWS} rows from {CSV_FILE} safely...")
raw_df = pd.read_csv(CSV_FILE, nrows=TARGET_ROWS)
raw_df.columns = raw_df.columns.str.strip()
raw_df.replace([np.inf, -np.inf], np.nan, inplace=True)
raw_df.dropna(inplace=True)

print(f"[DATA] Cleaned base shape: {raw_df.shape}.")

[DATA] Loading 250000 rows from friday.csv safely...


[DATA] Cleaned base shape: (250000, 91).


In [10]:
def find_column(df, patterns):
    for col in df.columns:
        for pattern in patterns:
            if pattern.lower() in col.lower():
                return col
    raise KeyError(f"Could not resolve column with patterns: {patterns}")

# Identify flow identifiers
flow_keys = ['Src IP', 'Dst IP', 'Src Port', 'Dst Port', 'Protocol']
matched_keys = [find_column(raw_df, [key]) for key in flow_keys]
label_col = find_column(raw_df, ['label'])

# Select non-trivial physical and behavioral footprint features (No Flag Counts!)
behavioral_features = [
    'Flow Duration',
    'Total Fwd Packet',
    'Total Bwd packets',
    'Flow Bytes/s',
    'Flow Packets/s'
]
matched_features = [find_column(raw_df, [feat]) for feat in behavioral_features]

# Apply log transform to mitigate high scale skewness
for feat in matched_features:
    raw_df[feat] = np.log1p(np.abs(raw_df[feat].values))

# Chronological alignment
raw_df.sort_index(inplace=True)

# Isolate benign traffic exclusively to build the reconstruction standard
benign_mask = raw_df[label_col].str.contains('BENIGN', case=False, na=False)

scaler = RobustScaler()
scaler.fit(raw_df.loc[benign_mask, matched_features])
scaled_data = scaler.transform(raw_df[matched_features])

for i, col_name in enumerate(matched_features):
    raw_df[col_name + '_scaled'] = scaled_data[:, i]

scaled_cols = [f + '_scaled' for f in matched_features]
print(f"[INFO] Selected Behavioral Core: {scaled_cols}")

[INFO] Selected Behavioral Core: ['Flow Duration_scaled', 'Total Fwd Packet_scaled', 'Total Bwd packets_scaled', 'Flow Bytes/s_scaled', 'Flow Packets/s_scaled']


In [11]:
TIME_STEPS = 5
X_sequences = []
y_sequences = []

group_cols = [raw_df[key].astype(str) for key in matched_keys]
flow_groups = raw_df.groupby(group_cols, sort=False)

for name, group_df in flow_groups:
    group_len = len(group_df)
    group_data = group_df[scaled_cols].values
    group_labels = group_df[label_col].str.contains('BENIGN', case=False, na=False).astype(int).values

    if group_len >= TIME_STEPS:
        # Construct standard sequence frames
        for j in range(group_len - TIME_STEPS + 1):
            X_sequences.append(group_data[j : j + TIME_STEPS])
            # Sequence is tagged as anomalous if any window packet is an anomaly
            window_label = 0 if np.all(group_labels[j : j + TIME_STEPS] == 1) else 1
            y_sequences.append(window_label)
    else:
        # Pad short-lived sequences to match the dimension window
        padding_len = TIME_STEPS - group_len
        padded_frame = np.vstack([group_data, np.zeros((padding_len, len(scaled_cols)))])
        X_sequences.append(padded_frame)
        window_label = 0 if np.all(group_labels == 1) else 1
        y_sequences.append(window_label)

X_flow = np.array(X_sequences)
y_flow = np.array(y_sequences)

normal_idx = np.where(y_flow == 0)[0]
anomaly_idx = np.where(y_flow == 1)[0]

print(f"[PREP] Created {len(X_flow)} grouped/padded sequences.")
print(f"[PREP] Benign Sequences: {len(normal_idx)} | Anomaly Sequences: {len(anomaly_idx)}")

[PREP] Created 183808 grouped/padded sequences.
[PREP] Benign Sequences: 119316 | Anomaly Sequences: 64492


## Extract and Split Benign Baseline Sequences

In [12]:
def extract_chaotic_signal_features(sequences):
    """Calculates statistical entropy, variance, and rolling standard deviation inside each sequence step."""
    entropy_features = []
    variance_features = []
    std_features = []
    for seq in sequences:
        v = np.var(seq, axis=0, keepdims=True)
        std = np.std(seq, axis=0, keepdims=True)
        seq_abs = np.abs(seq) + 1e-9
        prob = seq_abs / np.sum(seq_abs, axis=0, keepdims=True)
        ent = -np.sum(prob * np.log(prob + 1e-9), axis=0, keepdims=True)
        entropy_features.append(ent)
        variance_features.append(v)
        std_features.append(std)
    return np.concatenate(entropy_features, axis=0), np.concatenate(variance_features, axis=0), np.concatenate(std_features, axis=0)

# Splitting strictly benign sequences: 60% Train, 20% Validation, 20% Test to prevent data leakage
benign_indices = normal_idx.copy()
np.random.seed(42)
np.random.shuffle(benign_indices)

num_benign = len(benign_indices)
train_split = int(0.6 * num_benign)
val_split = int(0.8 * num_benign)

train_benign_idx = benign_indices[:train_split]
val_benign_idx = benign_indices[train_split:val_split]
test_benign_idx = benign_indices[val_split:]

X_train_benign = X_flow[train_benign_idx]
X_val_benign = X_flow[val_benign_idx]
X_test_benign_raw = X_flow[test_benign_idx]

# Construct mixed test set using the isolated 20% test benign sequences and equal amount of anomaly sequences
all_anomalies_raw = X_flow[anomaly_idx]
target_anomaly_count = len(X_test_benign_raw)

# Randomly sample anomalies matching the test benign count
np.random.seed(42)
anomaly_sample_indices = np.random.choice(len(all_anomalies_raw), target_anomaly_count, replace=False)
eval_anomaly_sampled_raw = all_anomalies_raw[anomaly_sample_indices]

X_eval_mix_raw = np.concatenate([X_test_benign_raw, eval_anomaly_sampled_raw], axis=0)
y_eval_mix = np.concatenate([np.zeros(len(X_test_benign_raw)), np.ones(target_anomaly_count)], axis=0)

# Compute physics and standard deviation features for evaluation set
mix_ent, mix_var, mix_std = extract_chaotic_signal_features(X_eval_mix_raw)
mix_ent_exp = np.repeat(mix_ent[:, np.newaxis, :], TIME_STEPS, axis=1)
mix_var_exp = np.repeat(mix_var[:, np.newaxis, :], TIME_STEPS, axis=1)
mix_std_exp = np.repeat(mix_std[:, np.newaxis, :], TIME_STEPS, axis=1)
X_eval_mix = np.concatenate([X_eval_mix_raw, mix_ent_exp, mix_var_exp, mix_std_exp], axis=-1)

print(f"[INFO] Train Benign: {len(X_train_benign)} | Val Benign: {len(X_val_benign)} | Test Benign: {len(X_test_benign_raw)}")

[INFO] Train Benign: 71589 | Val Benign: 23863 | Test Benign: 23864


## Model Compiling: Physics-Guided LSTM Autoencoder Architecture

In [13]:
def compile_chaotic_lstm_autoencoder(input_shape):
    inputs = Input(shape=input_shape)
    masked = Masking(mask_value=0.0)(inputs)
    encoder = LSTM(32, activation='tanh', return_sequences=False)(masked)
    replicated = RepeatVector(input_shape[0])(encoder)
    decoder = LSTM(32, activation='tanh', return_sequences=True)(replicated)
    outputs = TimeDistributed(Dense(input_shape[1]))(decoder)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mae')
    return model

## Baseline Training, Calibration & Evaluation

In [ ]:
# Extract physics-guided features plus standard deviation strictly on benign splits
benign_train_ent, benign_train_var, benign_train_std = extract_chaotic_signal_features(X_train_benign)
benign_val_ent, benign_val_var, benign_val_std = extract_chaotic_signal_features(X_val_benign)

# Match 3D sequence tensor dimensions (replicate features over TIME_STEPS)
train_ent_exp = np.repeat(benign_train_ent[:, np.newaxis, :], TIME_STEPS, axis=1)
train_var_exp = np.repeat(benign_train_var[:, np.newaxis, :], TIME_STEPS, axis=1)
train_std_exp = np.repeat(benign_train_std[:, np.newaxis, :], TIME_STEPS, axis=1)

val_ent_exp = np.repeat(benign_val_ent[:, np.newaxis, :], TIME_STEPS, axis=1)
val_var_exp = np.repeat(benign_val_var[:, np.newaxis, :], TIME_STEPS, axis=1)
val_std_exp = np.repeat(benign_val_std[:, np.newaxis, :], TIME_STEPS, axis=1)

# Construct chaotic baseline spaces with expanded features
X_train_benign_chaotic = np.concatenate([X_train_benign, train_ent_exp, train_var_exp, train_std_exp], axis=-1)
X_val_benign_chaotic = np.concatenate([X_val_benign, val_ent_exp, val_var_exp, val_std_exp], axis=-1)

print(f"[PREP] Benign Chaotic Train shape: {X_train_benign_chaotic.shape}")
print(f"[PREP] Benign Chaotic Validation shape: {X_val_benign_chaotic.shape}")

[PREP] Benign Chaotic Train shape: (71589, 5, 20)
[PREP] Benign Chaotic Validation shape: (23863, 5, 20)


In [ ]:
# Train the chaotic autoencoder strictly on Benign-only chaotic signal sequences
chaotic_shape = (TIME_STEPS, X_train_benign_chaotic.shape[2])
chaotic_ae = compile_chaotic_lstm_autoencoder(chaotic_shape)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Apply sample weighting during loss computation to avoid optimization bias on padded time-steps
sample_weights = np.any(X_train_benign_chaotic != 0.0, axis=-1).astype(np.float32)
val_sample_weights = np.any(X_val_benign_chaotic != 0.0, axis=-1).astype(np.float32)

print("[TRAIN] Training Chaotic Sequence Autoencoder on Benign Baseline...")
chaotic_ae.fit(
    X_train_benign_chaotic, X_train_benign_chaotic,
    epochs=10,
    batch_size=256,
    sample_weight=sample_weights,
    validation_data=(X_val_benign_chaotic, X_val_benign_chaotic, val_sample_weights),
    callbacks=[early_stop],
    verbose=1
)

# Predict on Benign Validation to calibrate reconstruction threshold
X_val_benign_pred = chaotic_ae.predict(X_val_benign_chaotic)
val_benign_mae = np.mean(np.abs(X_val_benign_pred - X_val_benign_chaotic), axis=(1, 2))

# Calibrate dynamic statistical threshold
mu = np.mean(val_benign_mae)
sigma = np.std(val_benign_mae)
optimal_threshold = mu + 0.1 * sigma
print(f"[CALIBRATION] Calibrated Optimal Dynamic Reconstruction Threshold: {optimal_threshold:.6f}")

[TRAIN] Training Chaotic Sequence Autoencoder on Benign Baseline...
Epoch 1/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 17s 32ms/step - loss: 0.2123 - val_loss: 0.1260
Epoch 2/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 11s 41ms/step - loss: 0.1115 - val_loss: 0.0973
Epoch 3/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - loss: 0.0893 - val_loss: 0.0811
Epoch 4/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - loss: 0.0767 - val_loss: 0.0708
Epoch 5/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - loss: 0.0682 - val_loss: 0.0651
Epoch 6/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - loss: 0.0631 - val_loss: 0.0608
Epoch 7/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - loss: 0.0593 - val_loss: 0.0577
Epoch 8/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - loss: 0.0566 - val_loss: 0.0548
Epoch 9/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 11s 33ms/step - loss: 0.0544 - val_loss: 0.0532
Epoch 10/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 12s 42ms/step - loss: 0.0522 - val_loss: 0.0521
746/746 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step
[CALIBRATION

In [16]:
# Reconstruction Anomaly Detection Evaluation
X_eval_preds = chaotic_ae.predict(X_eval_mix)
eval_mae = np.mean(np.abs(X_eval_preds - X_eval_mix), axis=(1, 2))
final_predictions = (eval_mae > optimal_threshold).astype(int)

print("\n Sequence Autoencoder Reconstruction Evaluation Summary ")
print(classification_report(y_eval_mix, final_predictions, target_names=['Benign', 'Anomaly'], zero_division=0))

1492/1492 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step

 Sequence Autoencoder Reconstruction Evaluation Summary 
              precision    recall  f1-score   support

      Benign       0.86      0.69      0.77     23864
     Anomaly       0.74      0.88      0.81     23864

    accuracy                           0.79     47728
   macro avg       0.80      0.79      0.79     47728
weighted avg       0.80      0.79      0.79     47728

